# STEP 1- Google Colab Setup for PySpark

1.1 Install JAVA & PySpark

In [ ]:
# Install Java and PySpark
!apt-get install openjdk-11-jdk -y
!pip install -q pyspark

# Set JAVA_HOME
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("AQI Enrichment").getOrCreate()

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra libatk-wrapper-java
  libatk-wrapper-java-jni libxt-dev libxtst6 libxxf86dga1 openjdk-11-jre
  x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm mesa-utils
The following NEW packages will be installed:
  fonts-dejavu-core fonts-dejavu-extra libatk-wrapper-java
  libatk-wrapper-java-jni libxt-dev libxtst6 libxxf86dga1 openjdk-11-jdk
  openjdk-11-jre x11-utils
0 upgraded, 10 newly installed, 0 to remove and 35 not upgraded.
Need to get 6,920 kB of archives.
After this operation, 16.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-core all 2.37-2build1 [1,041 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 fonts-dejavu-extra all 2.37-2build1 [2,041 kB]
Get:3 http://archive.ubuntu.com/ubuntu jam

6.2 Initialize SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Air Pollution Health Impact Analysis") \
    .getOrCreate()

spark

In [ ]:
spark.version

'3.5.1'

# STEP 2 - Load the Dataset into PySpark

2.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Read the CSV into PySpark DataFrame

In [ ]:
file_path = "/content/drive/MyDrive/Air pollution & health impact/city_day_clean.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)


Check the DataFrame Schema and Sample Data

In [1]:
df.printSchema()
df.show(5)


NameError: name 'df' is not defined

# Step 3: Understand and Clean the Dataset

3.1 DataTypes and column of our datasets


In [ ]:
df.printSchema()


root
 |-- City: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- PM25: double (nullable = true)
 |-- PM10: double (nullable = true)
 |-- NO: double (nullable = true)
 |-- NO2: double (nullable = true)
 |-- NOx: double (nullable = true)
 |-- NH3: double (nullable = true)
 |-- CO: double (nullable = true)
 |-- SO2: double (nullable = true)
 |-- O3: double (nullable = true)
 |-- Benzene: double (nullable = true)
 |-- Toluene: double (nullable = true)
 |-- Xylene: double (nullable = true)
 |-- AQI: integer (nullable = true)
 |-- AQI_Bucket: string (nullable = true)



3.2 Checking missing data in each column:

In [ ]:
from pyspark.sql.functions import col, isnan, when, count

df.select([count(when(col(c).isNull() | isnan(c), c)).alias(c) for c in df.columns]).show()


+----+----+----+-----+----+----+----+-----+----+----+----+-------+-------+------+----+----------+
|City|Date|PM25| PM10|  NO| NO2| NOx|  NH3|  CO| SO2|  O3|Benzene|Toluene|Xylene| AQI|AQI_Bucket|
+----+----+----+-----+----+----+----+-----+----+----+----+-------+-------+------+----+----------+
|   0|   0|4598|11140|3582|3585|4185|10328|2059|3854|4022|   5623|   8041| 18109|4681|      4681|
+----+----+----+-----+----+----+----+-----+----+----+----+-------+-------+------+----+----------+



# Step 4: Data Cleaning & Preprocessing

2.1. Drop rows with null values in critical columns

In [ ]:
df = df.dropna(subset=["City", "Date", "AQI", "AQI_Bucket"])


In [ ]:
from pyspark.sql.functions import to_date

df = df.withColumn("Date", to_date("Date", "M/d/yyyy"))


In [ ]:
from pyspark.sql.functions import year, month, date_format

df = df.withColumn("Year", year("Date"))
df = df.withColumn("Month", month("Date"))
df = df.withColumn("Weekday", date_format("Date", "EEEE"))


2.2. Fill NaN or null values in numeric columns using the mean

In [ ]:
from pyspark.sql.functions import avg

numeric_cols = ['PM25', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3']

for col_name in numeric_cols:
    mean_value = df.select(avg(f"`{col_name}`")).first()[0]
    df = df.fillna({col_name: mean_value})


In [ ]:
required_cols = ['City', 'Date', 'AQI', 'AQI_Bucket', 'PM25', 'PM10', 'NO2', 'SO2', 'CO', 'Year', 'Month']
df_cleaned_final = df.dropna(subset=required_cols)


In [ ]:
# Save only the cleaned data
df_cleaned_final.toPandas().to_csv("/content/drive/MyDrive/Air pollution & health impact/aqi_enriched_dataset.csv", index=False)


4.3. Convert Date column to PySpark DateType

In [ ]:
from pyspark.sql.functions import to_date

df = df.withColumn("Date", to_date("Date", "M/d/yyyy"))


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, year

# Start Spark session
spark = SparkSession.builder.appName("FixDateAndYear").getOrCreate()

# Load your manually cleaned file
df = spark.read.csv("/content/drive/MyDrive/Air pollution & health impact/city_day_clean.csv", header=True, inferSchema=True)

# ✅ Fix the Date using the correct format
df = df.withColumn("Date", to_date("Date", "M/d/yyyy"))

# ✅ Extract the Year
df = df.withColumn("Year", year("Date"))



# Save the clean file with Year extracted
df.toPandas().to_csv("/content/drive/MyDrive/Air pollution & health impact/aqi_by_city_year_FIXED_NT.csv", index=False)


4.4. Confirm Cleaned Data

In [ ]:
# Preview the fixed columns
df.select("Date", "Year","AQI","AQI_Bucket").show(10)



+----------+----+----+----------+
|      Date|Year| AQI|AQI_Bucket|
+----------+----+----+----------+
|2015-01-01|2015|NULL|      NULL|
|2015-01-02|2015|NULL|      NULL|
|2015-01-03|2015|NULL|      NULL|
|2015-01-04|2015|NULL|      NULL|
|2015-01-05|2015|NULL|      NULL|
|2015-01-06|2015|NULL|      NULL|
|2015-01-07|2015|NULL|      NULL|
|2015-01-08|2015|NULL|      NULL|
|2015-01-09|2015|NULL|      NULL|
|2015-01-10|2015|NULL|      NULL|
+----------+----+----+----------+
only showing top 10 rows



In [ ]:
df.printSchema()
df.show(5)


root
 |-- City: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- PM25: double (nullable = true)
 |-- PM10: double (nullable = true)
 |-- NO: double (nullable = true)
 |-- NO2: double (nullable = true)
 |-- NOx: double (nullable = true)
 |-- NH3: double (nullable = true)
 |-- CO: double (nullable = true)
 |-- SO2: double (nullable = true)
 |-- O3: double (nullable = true)
 |-- Benzene: double (nullable = true)
 |-- Toluene: double (nullable = true)
 |-- Xylene: double (nullable = true)
 |-- AQI: integer (nullable = true)
 |-- AQI_Bucket: string (nullable = true)
 |-- Year: integer (nullable = true)

+---------+----------+----+----+----+-----+-----+----+----+-----+------+-------+-------+------+----+----------+----+
|     City|      Date|PM25|PM10|  NO|  NO2|  NOx| NH3|  CO|  SO2|    O3|Benzene|Toluene|Xylene| AQI|AQI_Bucket|Year|
+---------+----------+----+----+----+-----+-----+----+----+-----+------+-------+-------+------+----+----------+----+
|Ahmedabad|2015-01-01|NULL|N

#Step 5: Export clean enriched CSV

In [ ]:

# Convert PySpark DataFrame to Pandas
df.toPandas().to_csv("aqi_enriched_dataset.csv", index=False)


from google.colab import files
files.download("aqi_enriched_dataset.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#Step 5: Exploratory Data Analysis (EDA)

*   List item
*   List item



5.1. Average AQI by City (Top 10 Most Polluted)

In [ ]:
df.groupBy("City") \
  .avg("AQI") \
  .withColumnRenamed("avg(AQI)", "Average_AQI") \
  .orderBy("Average_AQI", ascending=False) \
  .show(10)


+------------+------------------+
|        City|       Average_AQI|
+------------+------------------+
|   Ahmedabad|452.12293853073464|
|       Delhi|259.48774387193595|
|       Patna| 240.7820424948595|
|    Gurugram|225.12388162422573|
|     Lucknow|  217.973058637084|
|     Talcher|172.88681948424068|
|  Jorapokhar|159.25162127107652|
|Brajrajnagar| 150.2805049088359|
|     Kolkata|140.56631299734747|
|    Guwahati|140.11111111111111|
+------------+------------------+
only showing top 10 rows



5.2. Average AQI Over Time (Nationwide)

In [ ]:
df.groupBy("Year") \
  .avg("AQI") \
  .withColumnRenamed("avg(AQI)", "Average_AQI") \
  .orderBy("Year") \
  .show()



+----+------------------+
|Year|       Average_AQI|
+----+------------------+
|2015| 212.4630541871921|
|2016|  197.150019432569|
|2017|181.47278911564626|
|2018|182.68431167016072|
|2019|156.51817281855466|
|2020|113.52069667496042|
+----+------------------+



5.3. Average AQI Per City Per Year (for Tableau export)

In [ ]:
from pyspark.sql.functions import year

df_city_year = df_cleaned.withColumn("Year", year("Date")) \
    .groupBy("City", "Year") \
    .avg("AQI") \
    .withColumnRenamed("avg(AQI)", "Average_AQI")

df_city_year.show(10)


+------------+----+------------------+
|        City|Year|       Average_AQI|
+------------+----+------------------+
|       Kochi|NULL|104.28481012658227|
|       Patna|NULL| 240.7820424948595|
|   Ernakulam|NULL|   92.359477124183|
|     Chennai|NULL|114.50265392781316|
|     Lucknow|NULL|  217.973058637084|
|      Mumbai|NULL|105.35225806451614|
|      Aizawl|NULL|34.765765765765764|
|   Ahmedabad|NULL|452.12293853073464|
|    Gurugram|NULL|225.12388162422573|
|Brajrajnagar|NULL| 150.2805049088359|
+------------+----+------------------+
only showing top 10 rows



5.4 AQI Trend by City (City-Year Analysis)

In [ ]:
df.groupBy("City", "Year") \
  .avg("AQI") \
  .withColumnRenamed("avg(AQI)", "Average_AQI") \
  .orderBy("City", "Year") \
  .show(20)


+---------+----+------------------+
|     City|Year|       Average_AQI|
+---------+----+------------------+
|Ahmedabad|2015| 310.9505703422053|
|Ahmedabad|2016| 310.1623931623932|
|Ahmedabad|2017|  558.768115942029|
|Ahmedabad|2018| 622.2633053221289|
|Ahmedabad|2019| 516.3522727272727|
|Ahmedabad|2020| 242.0681818181818|
|   Aizawl|2020|34.765765765765764|
|Amaravati|2017|192.51351351351352|
|Amaravati|2018|101.39102564102564|
|Amaravati|2019| 98.48543689320388|
|Amaravati|2020| 59.87978142076503|
| Amritsar|2017|148.06766917293234|
| Amritsar|2018|122.92592592592592|
| Amritsar|2019|             109.5|
| Amritsar|2020| 92.97701149425288|
|Bengaluru|2015|112.57342657342657|
|Bengaluru|2016|105.58404558404558|
|Bengaluru|2017| 87.12087912087912|
|Bengaluru|2018| 86.30747922437673|
|Bengaluru|2019|  91.6027397260274|
+---------+----+------------------+
only showing top 20 rows



5.4. Correlation Between AQI and Pollutants (Optional)

In [ ]:
numeric_cols = ['PM25', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
for col_name in numeric_cols:
    corr = df.stat.corr("AQI", col_name)
    print(f"Correlation between AQI and {col_name}: {corr}")


Correlation between AQI and PM25: 0.6645907141394649
Correlation between AQI and PM10: 0.32155326123926375
Correlation between AQI and NO: 0.46450629293318535
Correlation between AQI and NO2: 0.5771515173324436
Correlation between AQI and NOx: 0.43183751379007596
Correlation between AQI and NH3: 0.07993945756131415
Correlation between AQI and CO: 0.6163778573292542
Correlation between AQI and SO2: 0.47195562840398136
Correlation between AQI and O3: 0.3177445859150738
Correlation between AQI and Benzene: 0.06589222160444633
Correlation between AQI and Toluene: 0.30760011401048604
Correlation between AQI and Xylene: 0.21036757200731726


5.5. Save Cleaned Data for Tableau Dashboard

In [ ]:
df.groupBy("City", "Year") \
  .avg("AQI") \
  .withColumnRenamed("avg(AQI)", "Average_AQI") \
  .toPandas() \
  .to_csv("/content/drive/MyDrive/Air pollution & health impact/aqi_by_city_year_FIXED.csv", index=False)


In [ ]:
df.groupBy("City", "Year").agg({"AQI": "avg"}).orderBy("City", "Year").show(50)


+------------+----+------------------+
|        City|Year|          avg(AQI)|
+------------+----+------------------+
|   Ahmedabad|2015| 310.9505703422053|
|   Ahmedabad|2016| 310.1623931623932|
|   Ahmedabad|2017|  558.768115942029|
|   Ahmedabad|2018| 622.2633053221289|
|   Ahmedabad|2019| 516.3522727272727|
|   Ahmedabad|2020| 242.0681818181818|
|      Aizawl|2020|34.765765765765764|
|   Amaravati|2017|192.51351351351352|
|   Amaravati|2018|101.39102564102564|
|   Amaravati|2019| 98.48543689320388|
|   Amaravati|2020| 59.87978142076503|
|    Amritsar|2017|148.06766917293234|
|    Amritsar|2018|122.92592592592592|
|    Amritsar|2019|             109.5|
|    Amritsar|2020| 92.97701149425288|
|   Bengaluru|2015|112.57342657342657|
|   Bengaluru|2016|105.58404558404558|
|   Bengaluru|2017| 87.12087912087912|
|   Bengaluru|2018| 86.30747922437673|
|   Bengaluru|2019|  91.6027397260274|
|   Bengaluru|2020| 79.71584699453553|
|      Bhopal|2019| 162.6095238095238|
|      Bhopal|2020| 114.7